# Persist Anomaly Alerts

Runs the Eventhouse anomaly detectors (`fn_new_anomaly_alerts()` in
`fabric/kql_database/06-ml-anomaly-detection.kql`) and appends new alerts to
`anomaly_alerts`, then records the run in `anomaly_detection_runs`.

## Behavior
- Only closed hourly bins from the last `ANOMALY_PERSIST_HOURS` hours are
  persisted; the open hour is skipped because its counts are partial.
- `alert_id` is deterministic per metric, series and hourly bin. The KQL function
  left-anti joins existing alerts, and the write is tagged with a hash of the
  alert ids, so a rerun or retry never duplicates an alert.
- Every run, including one that finds nothing, appends one row to
  `anomaly_detection_runs`. Readiness uses that row as detection evidence.

## Schedule
Runs inside the `streaming-data-load` pipeline, next to `03-streaming-to-silver`.

In [ ]:
import hashlib
import json
import os
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F


def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", "retail_lakehouse")
KQL_DATABASE = get_env("KQL_DATABASE", "retail_eventhouse")
KUSTO_URI = get_env("KUSTO_URI", "")
ANOMALY_LOOKBACK_HOURS = int(get_env("ANOMALY_LOOKBACK_HOURS", "168"))
ANOMALY_PERSIST_HOURS = int(get_env("ANOMALY_PERSIST_HOURS", "24"))

In [ ]:
if ANOMALY_LOOKBACK_HOURS < 24 or ANOMALY_PERSIST_HOURS < 1:
    raise ValueError("ANOMALY_LOOKBACK_HOURS must be >= 24 and ANOMALY_PERSIST_HOURS >= 1")
if ANOMALY_PERSIST_HOURS > ANOMALY_LOOKBACK_HOURS:
    raise ValueError("ANOMALY_PERSIST_HOURS cannot exceed ANOMALY_LOOKBACK_HOURS")

# Column order of anomaly_alerts in 06-ml-anomaly-detection.kql; the connector
# writes by position.
ALERT_COLUMNS = (
    "alert_id", "detection_time", "metric_type", "entity_type", "entity_id",
    "time_window", "expected_value", "actual_value", "anomaly_score", "severity",
    "alert_details", "ingest_timestamp",
)
RUN_SCHEMA = (
    "run_id string, started_at timestamp, completed_at timestamp, "
    "closed_bin_end timestamp, lookback_hours long, persist_hours long, "
    "appended_alerts long"
)


def closed_bin_end(now):
    """End of the newest closed hourly bin (the open hour is excluded)."""
    return now.astimezone(timezone.utc).replace(minute=0, second=0, microsecond=0)


def detection_query(window_end, lookback_hours, persist_hours):
    return (
        f"fn_new_anomaly_alerts({int(lookback_hours)}, {int(persist_hours)}, "
        f"datetime({window_end.strftime('%Y-%m-%dT%H:%M:%SZ')}))"
    )


def alert_batch_identity(alert_ids):
    """Stable tag for one set of new alerts, so a retried write is skipped."""
    digest = hashlib.sha256("\n".join(sorted(alert_ids)).encode("utf-8")).hexdigest()
    return f"anomaly-alerts:{digest}"

In [ ]:
# =============================================================================
# EVENTHOUSE ACCESS (Spark Kusto connector)
# =============================================================================

KUSTO_FORMAT = "com.microsoft.kusto.spark.synapse.datasource"


def resolve_kusto_uri(database_name):
    """Resolve the KQL database Query URI from the current Fabric workspace."""
    import sempy.fabric as fabric

    client = fabric.FabricRestClient()
    workspace_id = fabric.get_notebook_workspace_id()
    response = client.get(f"v1/workspaces/{workspace_id}/kqlDatabases")
    response.raise_for_status()
    databases = response.json().get("value", [])
    for item in databases:
        if item.get("displayName") == database_name:
            uri = (item.get("properties") or {}).get("queryServiceUri")
            if uri:
                return uri
    found = ", ".join(sorted(d.get("displayName", "?") for d in databases)) or "<none>"
    raise ValueError(
        f"Could not find a KQL database named {database_name!r} in this workspace "
        f"(found: {found}). Set KQL_DATABASE or KUSTO_URI."
    )


if not KUSTO_URI:
    KUSTO_URI = resolve_kusto_uri(KQL_DATABASE)
print(f"Eventhouse database {KQL_DATABASE!r} at {KUSTO_URI}")
token = notebookutils.credentials.getToken(KUSTO_URI)  # noqa: F821


def read_kusto(query):
    return (
        spark.read.format(KUSTO_FORMAT)
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KQL_DATABASE)
        .option("accessToken", token)
        .option("kustoQuery", query)
        .load()
    )


def append_kusto(frame, table, identity):
    """Append a frame by column position; the identity tag makes retries idempotent."""
    tag = f"retail-demo-job:{identity}"
    (
        frame.write.format(KUSTO_FORMAT)
        .option("kustoCluster", KUSTO_URI)
        .option("kustoDatabase", KQL_DATABASE)
        .option("kustoTable", table)
        .option("accessToken", token)
        .option("tableCreateOptions", "FailIfNotExist")
        .option("writeMode", "Transactional")
        .option("requestId", str(uuid.uuid5(uuid.NAMESPACE_URL, tag)))
        .option("ensureNoDuplicatedBlobs", "true")
        .option("sparkIngestionPropertiesJson", json.dumps({
            "flushImmediately": True,
            "ingestByTags": [tag],
            "ingestIfNotExists": [tag],
        }))
        .mode("Append")
        .save()
    )

In [ ]:
# =============================================================================
# DETECT AND PERSIST
# =============================================================================

started_at = datetime.now(timezone.utc)
window_end = closed_bin_end(started_at)
query = detection_query(window_end, ANOMALY_LOOKBACK_HOURS, ANOMALY_PERSIST_HOURS)
print(f"Running {query}")
candidates = read_kusto(query)
missing = [column for column in ALERT_COLUMNS if column not in candidates.columns]
if missing:
    raise RuntimeError(f"fn_new_anomaly_alerts() is missing columns {missing}; reapply 06 KQL.")
candidates = candidates.select(*ALERT_COLUMNS).persist()
try:
    alert_ids = [row.alert_id for row in candidates.select("alert_id").collect()]
    if len(alert_ids) != len(set(alert_ids)):
        raise RuntimeError("fn_new_anomaly_alerts() returned duplicate alert_id values.")
    if alert_ids:
        append_kusto(
            candidates.withColumn("alert_details", F.col("alert_details").cast("string")),
            "anomaly_alerts",
            alert_batch_identity(alert_ids),
        )
finally:
    candidates.unpersist()

completed_at = datetime.now(timezone.utc)
run_id = str(uuid.uuid4())
run = spark.createDataFrame(
    [(
        run_id, started_at, completed_at, window_end,
        ANOMALY_LOOKBACK_HOURS, ANOMALY_PERSIST_HOURS, len(alert_ids),
    )],
    RUN_SCHEMA,
)
append_kusto(run, "anomaly_detection_runs", f"anomaly-run:{run_id}")
print(
    f"Anomaly detection run {run_id}: {len(alert_ids)} new alert(s) for closed bins "
    f"before {window_end.isoformat()}."
)